# Task 2: Support Ticket Classification & Prioritization
**Future Interns ML Internship 2026**  
**Intern:** Akalya A | **CIN:** FIT/MAR26/ML6268

## Objective
Build an ML system that automatically classifies customer support tickets into categories (Billing, Technical, Account, General) and assigns priority levels (High / Medium / Low) using NLP.

In [ ]:
# Install required libraries (run once)
# !pip install pandas numpy scikit-learn matplotlib seaborn nltk

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import nltk
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import warnings
warnings.filterwarnings('ignore')

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt', quiet=True)

plt.style.use('seaborn-v0_8-darkgrid')
print('Libraries loaded successfully!')

## Step 1: Create / Load Dataset
Download from Kaggle: https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset  
Or use the synthetic dataset below for demonstration.

In [ ]:
# Synthetic realistic dataset (replace with Kaggle CSV for better results)
tickets_data = {
    'ticket_text': [
        # Billing - High
        'I was charged twice for my subscription this month please refund immediately',
        'Unauthorized charge on my credit card from your company urgent help needed',
        'My payment failed but money was deducted from my account',
        'Invoice shows wrong amount I need immediate correction',
        'Duplicate billing for annual plan please fix this now',
        # Billing - Medium
        'How do I update my billing address on the account',
        'Can I get a receipt for my last payment',
        'What payment methods do you accept for renewal',
        'I need to upgrade my plan and update payment details',
        'Please send me the invoice for December 2025',
        # Technical - High
        'Application crashes every time I try to login urgent production issue',
        'Complete system outage our team cannot work at all help immediately',
        'Data loss occurred after update all our files are missing critical',
        'API integration broken entire workflow is down need urgent fix',
        'Server error 500 on all requests application completely unusable',
        # Technical - Medium
        'The mobile app is slow and sometimes freezes on my phone',
        'Dashboard charts are not loading properly after browser update',
        'Export to CSV feature is not working as expected',
        'Notification emails are not being sent to users',
        'Search function returns incorrect results sometimes',
        # Account - High
        'My account has been hacked someone changed my password immediately help',
        'Cannot access account at all locked out need urgent assistance',
        'Suspicious login from unknown location need to secure account now',
        # Account - Medium
        'How do I change my account email address',
        'I forgot my password and the reset email is not arriving',
        'How to add a team member to my workspace',
        'I want to delete my account and all associated data',
        'Can I have two accounts with the same email',
        # General - Low
        'What are your business hours for customer support',
        'Do you offer a free trial for the premium plan',
        'Is there a mobile application available for Android',
        'What is the difference between basic and pro plans',
        'Where can I find your documentation and user guide',
        'Do you offer discounts for educational institutions',
        'Can I export my data before cancelling the subscription',
        'How many users can I add in the team plan',
    ],
    'category': [
        'Billing','Billing','Billing','Billing','Billing',
        'Billing','Billing','Billing','Billing','Billing',
        'Technical','Technical','Technical','Technical','Technical',
        'Technical','Technical','Technical','Technical','Technical',
        'Account','Account','Account',
        'Account','Account','Account','Account','Account',
        'General','General','General','General','General','General','General','General'
    ],
    'priority': [
        'High','High','High','High','High',
        'Medium','Medium','Medium','Medium','Low',
        'High','High','High','High','High',
        'Medium','Medium','Medium','Medium','Medium',
        'High','High','High',
        'Medium','Medium','Medium','Low','Low',
        'Low','Low','Low','Low','Low','Low','Low','Low'
    ]
}

df = pd.DataFrame(tickets_data)
print(f'Dataset size: {len(df)} tickets')
print('\nCategory distribution:')
print(df['category'].value_counts())
print('\nPriority distribution:')
print(df['priority'].value_counts())

## Step 2: Text Cleaning & Preprocessing

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
# Keep negation words — they matter for classification
keep = {'not', 'no', 'never', 'cannot', 'cant', "can't"}
stop_words = stop_words - keep

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)           # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

df['clean_text'] = df['ticket_text'].apply(preprocess)

print('Sample cleaned tickets:')
for i in range(3):
    print(f'  Original : {df["ticket_text"].iloc[i]}')
    print(f'  Cleaned  : {df["clean_text"].iloc[i]}')
    print()

## Step 3: Feature Extraction (TF-IDF)

In [ ]:
tfidf = TfidfVectorizer(max_features=500, ngram_range=(1, 2))
X = tfidf.fit_transform(df['clean_text'])
y_cat = df['category']
y_pri = df['priority']

print(f'TF-IDF matrix shape: {X.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_)}')

# Split data
X_train, X_test, y_cat_train, y_cat_test, y_pri_train, y_pri_test = train_test_split(
    X, y_cat, y_pri, test_size=0.25, random_state=42, stratify=y_cat
)
print(f'\nTraining samples: {X_train.shape[0]}')
print(f'Testing samples: {X_test.shape[0]}')

## Step 4: Train Classification Models

In [ ]:
# --- Category Classifier ---
lr_cat = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_cat.fit(X_train, y_cat_train)
cat_preds = lr_cat.predict(X_test)

print('CATEGORY CLASSIFICATION RESULTS')
print('='*45)
print(f'Accuracy: {accuracy_score(y_cat_test, cat_preds):.2%}')
print()
print(classification_report(y_cat_test, cat_preds))

In [ ]:
# --- Priority Classifier ---
lr_pri = LogisticRegression(max_iter=1000, C=1.0, random_state=42)
lr_pri.fit(X_train, y_pri_train)
pri_preds = lr_pri.predict(X_test)

print('PRIORITY CLASSIFICATION RESULTS')
print('='*45)
print(f'Accuracy: {accuracy_score(y_pri_test, pri_preds):.2%}')
print()
print(classification_report(y_pri_test, pri_preds))

## Step 5: Visualizations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Support Ticket Classification Dashboard', fontsize=15, fontweight='bold')

# Plot 1: Category distribution
cat_counts = df['category'].value_counts()
colors = ['#4e9af1', '#f1734e', '#4ef1a0', '#f1e94e']
axes[0].pie(cat_counts.values, labels=cat_counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90)
axes[0].set_title('Ticket Category Distribution')

# Plot 2: Priority distribution
pri_counts = df['priority'].value_counts()
priority_colors = {'High': '#e74c3c', 'Medium': '#f39c12', 'Low': '#27ae60'}
p_colors = [priority_colors[p] for p in pri_counts.index]
axes[1].bar(pri_counts.index, pri_counts.values, color=p_colors, edgecolor='black', alpha=0.8)
axes[1].set_title('Ticket Priority Distribution')
axes[1].set_ylabel('Number of Tickets')
for i, v in enumerate(pri_counts.values):
    axes[1].text(i, v + 0.2, str(v), ha='center', fontweight='bold')

# Plot 3: Confusion matrix for category
cm = confusion_matrix(y_cat_test, cat_preds, labels=lr_cat.classes_)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=lr_cat.classes_, yticklabels=lr_cat.classes_, ax=axes[2])
axes[2].set_title('Category Classifier — Confusion Matrix')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.savefig('task2_ticket_classification.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved as task2_ticket_classification.png')

## Step 6: Live Prediction Function

In [ ]:
def classify_ticket(ticket_text):
    """Classifies a new ticket and returns category + priority."""
    cleaned = preprocess(ticket_text)
    vec = tfidf.transform([cleaned])
    category = lr_cat.predict(vec)[0]
    priority = lr_pri.predict(vec)[0]
    cat_conf = lr_cat.predict_proba(vec).max()
    pri_conf = lr_pri.predict_proba(vec).max()
    print(f'Ticket  : "{ticket_text}"')
    print(f'Category: {category} (confidence: {cat_conf:.0%})')
    print(f'Priority: {priority} (confidence: {pri_conf:.0%})')
    print('-' * 50)

# Test on new tickets
classify_ticket("I cannot login and my presentation is in 30 minutes please help")
classify_ticket("Where can I find the user manual PDF")
classify_ticket("I was charged $99 but my plan is only $9")

## Conclusion
This notebook built a complete support ticket classification system:
- Preprocessed raw ticket text using NLTK (lemmatization, stopword removal)
- Converted text to numerical features using TF-IDF
- Trained Logistic Regression classifiers for both category and priority
- Evaluated with accuracy, precision, recall, and confusion matrices
- Built a reusable `classify_ticket()` prediction function

**Business Impact:** Reduces manual triage time, routes urgent tickets instantly, improves customer satisfaction.

**Dataset:** Customer Support Ticket Dataset (Kaggle)  
**Libraries:** pandas, nltk, scikit-learn, matplotlib, seaborn